# **7. Validación Final y Comparación de los 4 Modelos Supervisados vs. Línea Base**
**Tesis de Maestría en Ingeniería - Ingeniería de Sistemas y Computación (MLDS)**  
*Universidad Nacional de Colombia*

---

### **Objetivo del Cuaderno**
Consolidar y comparar de manera rigurosa el desempeño de los **4 modelos supervisados entrenados** frente a la **Línea Base Regulatoria (`BENCH_KWH_DIA`)** sobre las particiones de **Prueba (`b333`–`d998`)** y **Validación (`d999`–`ffff`, *Hold-out* intocado)**:
1. **Arquitectura 1:** `MLP Sequential` (Red densa tabular + *One-Hot*).
2. **Arquitectura 2:** `Embeddings + Wide & Deep Residual` (Ruta lineal + bloques residuales + *Entity Embeddings*).
3. **Arquitectura 3:** `Híbrida Conv1D-LSTM Hurdle` (Secuencia 12 meses + dos cabezas: `prob_cero` y `kwh_dia`).
4. **Modelo 4:** `LightGBM / GBM` (Árboles potenciados por gradiente con manejo nativo de nulos y categorías).
5. **Referencia Regulatoria:** `BENCH_KWH_DIA` (`PROM_SEM_BENCHMARK / DIAS_FACTURADOS`).


In [ ]:
# ==============================================================================
# 1. CONFIGURACIÓN DEL ENTORNO, RUTAS Y SEMILLAS
# ==============================================================================
import os
import glob
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

RUTA_BASE = '/content/drive/MyDrive/Tesis_Consumo_Bogota'
RUTA_DATOS = os.path.join(RUTA_BASE, 'Datos_Preprocesados')
RUTA_RES_5 = os.path.join(RUTA_BASE, 'Resultados_5_Entrenamiento_Modelos')
RUTA_RES_6 = os.path.join(RUTA_BASE, 'Resultados_6_LightGBM')
RUTA_SALIDA = os.path.join(RUTA_BASE, 'Resultados_7_Validacion')
os.makedirs(RUTA_SALIDA, exist_ok=True)

SEMILLA = 42
np.random.seed(SEMILLA)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3
})

PALETA_MODELOS = {
    'BENCH_KWH_DIA': '#7f8c8d',
    'Arq1_MLP_Sequential': '#34495e',
    'Arq2_Embeddings_WideDeep': '#1f77b4',
    'Arq3_Conv1D_LSTM_Hurdle': '#2ca02c',
    'Mod4_LightGBM': '#d62728'
}

NOMBRES_ETIQUETAS = {
    'BENCH_KWH_DIA': '0. Línea Base (Promedio Semestral)',
    'Arq1_MLP_Sequential': '1. Arq. 1: MLP Sequential',
    'Arq2_Embeddings_WideDeep': '2. Arq. 2: Wide & Deep + Embeddings',
    'Arq3_Conv1D_LSTM_Hurdle': '3. Arq. 3: Conv1D-LSTM Hurdle',
    'Mod4_LightGBM': '4. Modelo 4: LightGBM (GBM)'
}

print(f"Directorio de salida configurado en: {RUTA_SALIDA}")


In [ ]:
# ==============================================================================
# 2. CARGA DE DATOS DE VALIDACIÓN Y CACHÉ DE PREDICCIONES DE LOS 4 MODELOS
# ==============================================================================
def cargar_tabla_validacion_y_predicciones(particion='VALIDACION'):
    """
    Carga el vector objetivo real (KWH_DIA), la línea base (BENCH_KWH_DIA),
    metadatos por registro (NIU_USUARIO, ESTRATO_SECTOR, PERIODO, BENCH_VALIDO)
    y las predicciones en caché de los 4 modelos (Cuadernos 5 y 6).
    """
    p = particion.lower()
    ruta_meta = os.path.join(RUTA_DATOS, f'meta_{p}.parquet')
    
    if os.path.exists(ruta_meta):
        df = pd.read_parquet(ruta_meta)
    else:
        raise FileNotFoundError(
            f"No se encontró {ruta_meta}. Verifica la ruta de los bloques preprocesados."
        )

    # Carga de predicciones en caché generadas por Cuaderno 5 (Redes) y Cuaderno 6 (GBM)
    archivos_pred = {
        'Arq1_MLP_Sequential': [
            os.path.join(RUTA_RES_5, 'cache_predicciones', f'pred_arq1_mlp_sequential_{p}.npy'),
            os.path.join(RUTA_RES_5, f'pred_Arq1_MLP_Sequential_{p}.npy')
        ],
        'Arq2_Embeddings_WideDeep': [
            os.path.join(RUTA_RES_5, 'cache_predicciones', f'pred_arq2_embeddings_wide_deep_{p}.npy'),
            os.path.join(RUTA_RES_5, f'pred_Arq2_Embeddings_WideDeep_{p}.npy')
        ],
        'Arq3_Conv1D_LSTM_Hurdle': [
            os.path.join(RUTA_RES_5, 'cache_predicciones', f'pred_arq3_conv1d_lstm_hurdle_{p}.npy'),
            os.path.join(RUTA_RES_5, f'pred_Arq3_Conv1D_LSTM_Hurdle_{p}.npy')
        ],
        'Mod4_LightGBM': [
            os.path.join(RUTA_RES_6, 'cache_predicciones', f'pred_lightgbm_{p}.npy'),
            os.path.join(RUTA_RES_6, f'pred_Mod4_LightGBM_{p}.npy')
        ]
    }

    for mod, rutas in archivos_pred.items():
        cargado = False
        for r in rutas:
            if os.path.exists(r):
                df[mod] = np.clip(np.load(r).astype(np.float32).ravel(), 0.0, None)
                cargado = True
                print(f"[{particion}] Predicciones cargadas: {mod} ({len(df[mod]):,} filas)")
                break
        if not cargado:
            print(f"[ADVERTENCIA] No se encontró caché para {mod} en {particion}.")

    # Carga opcional de la probabilidad de cero de la Arq. 3
    for r_prob in [
        os.path.join(RUTA_RES_5, 'cache_predicciones', f'prob_cero_arq3_{p}.npy'),
        os.path.join(RUTA_RES_5, f'prob_cero_Arq3_{p}.npy')
    ]:
        if os.path.exists(r_prob):
            df['PROB_CERO_ARQ3'] = np.load(r_prob).astype(np.float32).ravel()
            print(f"[{particion}] Probabilidad de cero Arq. 3 cargada.")
            break

    return df

# Consolidación de tablas de búsqueda/prueba exportadas previamente (si existen)
def consolidar_resumen_prueba():
    tablas = []
    for ruta_csv in [
        os.path.join(RUTA_RES_5, '03_comparacion_modelos_prueba.csv'),
        os.path.join(RUTA_RES_5, '10_mejores_hiperparametros.csv'),
        os.path.join(RUTA_RES_6, '03_comparacion_gbm_prueba.csv')
    ]:
        if os.path.exists(ruta_csv):
            tablas.append(pd.read_csv(ruta_csv))
    if tablas:
        df_res = pd.concat(tablas, ignore_index=True)
        display(df_res)
        return df_res
    return None

df_resumen_previo = consolidar_resumen_prueba()


In [ ]:
# ==============================================================================
# 3. FUNCIONES DE MÉTRICAS E INFERENCIA POR SUBCONJUNTOS (SIN DESBORDAR RAM)
# ==============================================================================
def calcular_metricas_vector(y_true, y_pred):
    """Calcula MAE, RMSE, R2, MedAE y Sesgo en una sola pasada vectorizada en float64."""
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = y_true[mask].astype(np.float64)
    yp = y_pred[mask].astype(np.float64)
    n = yt.size
    if n == 0:
        return {'n': 0, 'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan, 'MedAE': np.nan, 'Sesgo': np.nan}

    err = yp - yt
    abs_err = np.abs(err)
    mae = float(np.mean(abs_err))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    sesgo = float(np.mean(err))
    medae = float(np.median(abs_err))

    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {
        'n': int(n),
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'MedAE': medae,
        'Sesgo': sesgo
    }

def evaluar_modelos_por_subconjunto(df, modelos):
    """
    Evalúa la Línea Base y los 4 modelos en 4 subconjuntos clave:
    1. Universo completo de validación.
    2. Subconjunto comparable con Benchmark válido (excluye sep-2023 y BENCH > 100 kWh/día).
    3. Meses de consumo real cero (KWH_DIA == 0: desocupación / suspensión).
    4. Meses de consumo positivo (KWH_DIA > 0).
    """
    y_real = df['KWH_DIA'].to_numpy(dtype=np.float32)
    bench = df['BENCH_KWH_DIA'].to_numpy(dtype=np.float32)

    if 'BENCH_VALIDO' in df.columns:
        mask_bench_val = (df['BENCH_VALIDO'].to_numpy() == 1) & (bench <= 100.0) & (y_real <= 100.0)
    else:
        mask_bench_val = (bench <= 100.0) & (y_real <= 100.0) & (df['PERIODO'].astype(str) != '2023-09-01')

    subconjuntos = {
        '1. Todos los registros': np.ones(len(df), dtype=bool),
        '2. Comparable con Benchmark (<=100 y sin sep-2023)': mask_bench_val,
        '3. Consumo real CERO (KWH_DIA = 0)': (y_real == 0.0) & mask_bench_val,
        '4. Consumo real POSITIVO (KWH_DIA > 0)': (y_real > 0.0) & mask_bench_val
    }

    filas = []
    for nombre_sub, mask in subconjuntos.items():
        mae_bench = calcular_metricas_vector(y_real[mask], bench[mask])['MAE']
        for mod in modelos:
            if mod not in df.columns:
                continue
            yp = df[mod].to_numpy(dtype=np.float32)
            m = calcular_metricas_vector(y_real[mask], yp[mask])
            mejora_pct = 100.0 * (mae_bench - m['MAE']) / mae_bench if mae_bench > 0 else np.nan
            filas.append({
                'Subconjunto': nombre_sub,
                'Código_Modelo': mod,
                'Modelo': NOMBRES_ETIQUETAS.get(mod, mod),
                'n': m['n'],
                'MAE': m['MAE'],
                'RMSE': m['RMSE'],
                'R2': m['R2'],
                'MedAE': m['MedAE'],
                'Sesgo': m['Sesgo'],
                'Mejora_MAE_vs_Bench_%': mejora_pct
            })

    return pd.DataFrame(filas)


In [ ]:
# ==============================================================================
# 4. EJECUCIÓN DE LA EVALUACIÓN COMPARATIVA EN VALIDACIÓN (HOLD-OUT)
# ==============================================================================
df_val = cargar_tabla_validacion_y_predicciones('VALIDACION')

MODELOS_EVALUAR = [
    'BENCH_KWH_DIA',
    'Arq1_MLP_Sequential',
    'Arq2_Embeddings_WideDeep',
    'Arq3_Conv1D_LSTM_Hurdle',
    'Mod4_LightGBM'
]
MODELOS_PRESENTES = [m for m in MODELOS_EVALUAR if m in df_val.columns]

tabla_global = evaluar_modelos_por_subconjunto(df_val, MODELOS_PRESENTES)
ruta_csv_global = os.path.join(RUTA_SALIDA, '01_comparacion_4_modelos_validacion.csv')
tabla_global.to_csv(ruta_csv_global, index=False)

print("\n=== TABLA COMPARATIVA GLOBAL EN VALIDACIÓN (HOLD-OUT) ===")
display(
    tabla_global.style.format({
        'n': '{:,}',
        'MAE': '{:.4f}',
        'RMSE': '{:.4f}',
        'R2': '{:.4f}',
        'MedAE': '{:.4f}',
        'Sesgo': '{:+.4f}',
        'Mejora_MAE_vs_Bench_%': '{:+.2f}%'
    }).highlight_min(subset=['MAE', 'RMSE'], color='#d4efdf')
      .highlight_max(subset=['R2', 'Mejora_MAE_vs_Bench_%'], color='#d4efdf')
)


In [ ]:
# ==============================================================================
# 5. BOOTSTRAP AGRUPADO POR USUARIO (NIU_USUARIO) - IC 95% Y PRUEBA PAREADA
# ==============================================================================
def bootstrap_agrupado_por_usuario(
    codigos_usuario, y_true, dict_predicciones, ref_key='BENCH_KWH_DIA',
    n_boot=1000, seed=42
):
    """
    Ejecuta Bootstrap por clúster de usuario (NIU_USUARIO) con complejidad O(N)
    en pre-agregación y O(B * U) en remuestreo mediante np.bincount.
    """
    rng = np.random.default_rng(seed)
     _, inv_idx = np.unique(codigos_usuario, return_inverse=True)
    n_usuarios = int(inv_idx.max() + 1)

    conteo_u = np.bincount(inv_idx, minlength=n_usuarios).astype(np.float64)
    sumas_err_u = {}
    for mod, y_pred in dict_predicciones.items():
        abs_e = np.abs(y_true - y_pred).astype(np.float64)
        sumas_err_u[mod] = np.bincount(inv_idx, weights=abs_e, minlength=n_usuarios)

    resultados_boot = {mod: np.empty(n_boot, dtype=np.float64) for mod in dict_predicciones}

    for b in range(n_boot):
         muestra_u = rng.integers(0, n_usuarios, size=n_usuarios)
         pesos_u = np.bincount(muestra_u, minlength=n_usuarios).astype(np.float64)
         denom = float(np.dot(pesos_u, conteo_u))
         for mod in dict_predicciones:
             resultados_boot[mod][b] = float(np.dot(pesos_u, sumas_err_u[mod])) / denom

    # Identificar el mejor modelo en MAE puntual
    maes_puntuales = {
        mod: float(sumas_err_u[mod].sum() / conteo_u.sum())
        for mod in dict_predicciones if mod != ref_key
    }
    mejor_mod = min(maes_puntuales, key=maes_puntuales.get)

    filas_ic = []
    for mod in dict_predicciones:
        serie_mae = resultados_boot[mod]
        diff_vs_bench = resultados_boot[ref_key] - serie_mae
        diff_vs_mejor = serie_mae - resultados_boot[mejor_mod]

        filas_ic.append({
            'Modelo': NOMBRES_ETIQUETAS.get(mod, mod),
            'MAE_Puntual': float(np.mean(serie_mae)),
            'MAE_IC95_Inf': float(np.percentile(serie_mae, 2.5)),
            'MAE_IC95_Sup': float(np.percentile(serie_mae, 97.5)),
            'Ganancia_vs_Bench': float(np.mean(diff_vs_bench)),
            'Ganancia_Bench_IC95_Inf': float(np.percentile(diff_vs_bench, 2.5)),
            'Ganancia_Bench_IC95_Sup': float(np.percentile(diff_vs_bench, 97.5)),
            'Delta_vs_Mejor_Modelo': float(np.mean(diff_vs_mejor)),
            'Delta_Mejor_IC95_Inf': float(np.percentile(diff_vs_mejor, 2.5)),
            'Delta_Mejor_IC95_Sup': float(np.percentile(diff_vs_mejor, 97.5)),
            'p_val_emp_vs_Bench': float(np.mean(diff_vs_bench <= 0.0)) if mod != ref_key else np.nan
        })

    return pd.DataFrame(filas_ic), resultados_boot

# Filtrar al subconjunto comparable para la inferencia frente al Benchmark
mask_comp = (df_val['BENCH_KWH_DIA'] <= 100.0) & (df_val['KWH_DIA'] <= 100.0)
if 'BENCH_VALIDO' in df_val.columns:
    mask_comp &= (df_val['BENCH_VALIDO'] == 1)

dict_preds_comp = {
    m: df_val.loc[mask_comp, m].to_numpy(dtype=np.float32)
    for m in MODELOS_PRESENTES
}

t0 = time.time()
df_bootstrap, dist_bootstrap = bootstrap_agrupado_por_usuario(
    codigos_usuario=df_val.loc[mask_comp, 'NIU_USUARIO'].to_numpy(),
    y_true=df_val.loc[mask_comp, 'KWH_DIA'].to_numpy(dtype=np.float32),
    dict_predicciones=dict_preds_comp,
    ref_key='BENCH_KWH_DIA',
    n_boot=1000,
    seed=SEMILLA
)
print(f"Bootstrap de 1.000 réplicas por usuario completado en {time.time() - t0:.2f} s")

df_bootstrap.to_csv(os.path.join(RUTA_SALIDA, '02_bootstrap_ic95_validacion.csv'), index=False)
display(df_bootstrap.style.format(precision=4))


In [ ]:
# ==============================================================================
# 6. DESEMPEÑO DESGLOSADO POR ESTRATO SOCIOECONÓMICO (1 A 6) Y RÉGIMEN DE CONSUMO
# ==============================================================================
def evaluar_por_estrato(df, mask_base, modelos):
    df_sub = df.loc[mask_base]
    filas = []
    for estrato in sorted(df_sub['ESTRATO_SECTOR'].dropna().unique()):
        m_est = (df_sub['ESTRATO_SECTOR'] == estrato).to_numpy()
        yt = df_sub.loc[m_est, 'KWH_DIA'].to_numpy(dtype=np.float32)
        mae_b = calcular_metricas_vector(yt, df_sub.loc[m_est, 'BENCH_KWH_DIA'].to_numpy(dtype=np.float32))['MAE']

        for mod in modelos:
            yp = df_sub.loc[m_est, mod].to_numpy(dtype=np.float32)
            met = calcular_metricas_vector(yt, yp)
            filas.append({
                'Estrato': int(estrato),
                'Código_Modelo': mod,
                'Modelo': NOMBRES_ETIQUETAS.get(mod, mod),
                'n': met['n'],
                'MAE': met['MAE'],
                'RMSE': met['RMSE'],
                'R2': met['R2'],
                'Sesgo': met['Sesgo'],
                'Mejora_vs_Bench_%': 100.0 * (mae_b - met['MAE']) / mae_b
            })
    return pd.DataFrame(filas)

df_estrato = evaluar_por_estrato(df_val, mask_comp, MODELOS_PRESENTES)
df_estrato.to_csv(os.path.join(RUTA_SALIDA, '03_desempeno_por_estrato_validacion.csv'), index=False)

pivote_mae_estrato = df_estrato.pivot(index='Estrato', columns='Modelo', values='MAE')
print("\n=== MAE (kWh/día) POR ESTRATO SOCIOECONÓMICO EN VALIDACIÓN ===")
display(pivote_mae_estrato.style.format('{:.4f}').highlight_min(axis=1, color='#d4efdf'))


In [ ]:
# ==============================================================================
# 7. ANÁLISIS EN EL UMBRAL DE SUBSISTENCIA (130 kWh/mes ≈ 4,33 kWh/día) Y CEROS
# ==============================================================================
UMBRAL_SUBSISTENCIA_DIA = 130.0 / 30.0  # 4.3333 kWh/día

df_val['TRAMO_REGULATORIO'] = pd.cut(
    df_val['KWH_DIA'],
    bins=[-0.001, 0.0, 2.0, UMBRAL_SUBSISTENCIA_DIA, 8.0, 100.0],
    labels=[
        '0. Sin consumo (0 kWh/d)',
        '1. Bajo (0 - 2 kWh/d)',
        '2. Subsistencia (2 - 4,33 kWh/d)',
        '3. Medio-Alto (4,33 - 8 kWh/d)',
        '4. Alto (> 8 kWh/d)'
    ]
)

filas_tramo = []
for tramo, grupo in df_val.loc[mask_comp].groupby('TRAMO_REGULATORIO', observed=True):
    yt = grupo['KWH_DIA'].to_numpy(dtype=np.float32)
    for mod in MODELOS_PRESENTES:
        yp = grupo[mod].to_numpy(dtype=np.float32)
        met = calcular_metricas_vector(yt, yp)
        filas_tramo.append({
            'Tramo_Consumo': str(tramo),
            'Modelo': NOMBRES_ETIQUETAS.get(mod, mod),
            'n': met['n'],
            'MAE': met['MAE'],
            'RMSE': met['RMSE'],
            'Sesgo': met['Sesgo']
        })

df_tramos = pd.DataFrame(filas_tramo)
df_tramos.to_csv(os.path.join(RUTA_SALIDA, '04_desempeno_por_tramo_consumo.csv'), index=False)
display(df_tramos.pivot(index='Tramo_Consumo', columns='Modelo', values='MAE').style.format('{:.4f}').highlight_min(axis=1, color='#d4efdf'))


In [ ]:
# ==============================================================================
# 8. EVALUACIÓN DE ENSAMBLE PONDERADO (ARQ. 2 WIDE & DEEP + MODELO 4 LIGHTGBM)
# ==============================================================================
if 'Arq2_Embeddings_WideDeep' in df_val.columns and 'Mod4_LightGBM' in df_val.columns:
    yt_c = df_val.loc[mask_comp, 'KWH_DIA'].to_numpy(dtype=np.float32)
    p_arq2 = df_val.loc[mask_comp, 'Arq2_Embeddings_WideDeep'].to_numpy(dtype=np.float32)
    p_gbm = df_val.loc[mask_comp, 'Mod4_LightGBM'].to_numpy(dtype=np.float32)

    pesos_alpha = np.linspace(0.0, 1.0, 11)
    filas_ens = []
    for alpha in pesos_alpha:
        p_mix = alpha * p_arq2 + (1.0 - alpha) * p_gbm
        m_mix = calcular_metricas_vector(yt_c, p_mix)
        filas_ens.append({
            'Peso_Arq2_WideDeep': round(float(alpha), 2),
            'Peso_Mod4_LightGBM': round(float(1.0 - alpha), 2),
            'MAE': m_mix['MAE'],
            'RMSE': m_mix['RMSE'],
            'R2': m_mix['R2'],
            'Sesgo': m_mix['Sesgo']
        })
    df_ensamble = pd.DataFrame(filas_ens)
    df_ensamble.to_csv(os.path.join(RUTA_SALIDA, '05_barrido_ensamble_arq2_gbm.csv'), index=False)
    print("\n=== BARRIDO DE ENSAMBLE LINEAL: ARQ. 2 (WIDE & DEEP) + MODELO 4 (LIGHTGBM) ===")
    display(df_ensamble.style.format('{:.4f}').highlight_min(subset=['MAE', 'RMSE'], color='#d4efdf'))


In [ ]:
# ==============================================================================
# 9. GENERACIÓN DE FIGURAS DE CALIDAD PUBLICABLE PARA EL CAPÍTULO 4 DE LA TESIS
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: MAE global e IC 95% Bootstrap en Validación
df_plot_b = df_bootstrap.sort_values('MAE_Puntual', ascending=True)
y_pos = np.arange(len(df_plot_b))
err_inf = df_plot_b['MAE_Puntual'] - df_plot_b['MAE_IC95_Inf']
err_sup = df_plot_b['MAE_IC95_Sup'] - df_plot_b['MAE_Puntual']

axes[0].barh(
    y_pos, df_plot_b['MAE_Puntual'],
    xerr=[err_inf, err_sup], capsize=4,
    color=['#1f77b4', '#d62728', '#34495e', '#2ca02c', '#7f8c8d'][:len(df_plot_b)],
    alpha=0.88
)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(df_plot_b['Modelo'])
axes[0].invert_yaxis()
axes[0].set_xlabel('MAE en Validación (kWh/día) con IC 95% Bootstrap por usuario')
axes[0].set_title('A. Comparación Global de MAE (Partición Validación)')
for i, v in enumerate(df_plot_b['MAE_Puntual']):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

# Panel B: MAE por Estrato Socioeconómico (1 a 6)
for mod in MODELOS_PRESENTES:
    sub_e = df_estrato[df_estrato['Código_Modelo'] == mod]
    axes[1].plot(
        sub_e['Estrato'], sub_e['MAE'],
        marker='o', linewidth=2,
        color=PALETA_MODELOS.get(mod, '#333333'),
        label=NOMBRES_ETIQUETAS.get(mod, mod)
    )

axes[1].set_xlabel('Estrato Socioeconómico (Bogotá D.C.)')
axes[1].set_ylabel('MAE (kWh/día)')
axes[1].set_title('B. Error Absoluto Medio por Estrato en Validación')
axes[1].set_xticks([1, 2, 3, 4, 5, 6])
axes[1].legend(loc='upper left', fontsize=8.5)

plt.tight_layout()
ruta_fig1 = os.path.join(RUTA_SALIDA, 'fig01_comparacion_4_modelos_validacion.png')
plt.savefig(ruta_fig1, bbox_inches='tight')
plt.show()
print(f"Figura exportada en: {ruta_fig1}")


In [ ]:
# ==============================================================================
# 10. EXPORTACIÓN AUTOMÁTICA DE TABLAS EN FORMATO LATEX (LISTAS PARA EL PDF)
# ==============================================================================
tabla_comp_tex = tabla_global[
    tabla_global['Subconjunto'] == '2. Comparable con Benchmark (<=100 y sin sep-2023)'
][['Modelo', 'n', 'MAE', 'RMSE', 'R2', 'MedAE', 'Sesgo', 'Mejora_MAE_vs_Bench_%']].copy()

latex_str = tabla_comp_tex.to_latex(
    index=False,
    float_format="%.4f",
    caption="Desempeño comparativo de los cuatro modelos supervisados frente a la línea base semestral en la partición de validación.",
    label="tab:comparacion_final_validacion"
)

ruta_tex = os.path.join(RUTA_SALIDA, 'tabla_4_comparacion_validacion.tex')
with open(ruta_tex, 'w', encoding='utf-8') as f:
    f.write(latex_str)

print("=== CÓDIGO LATEX GENERADO PARA EL CAPÍTULO 4 ===")
print(latex_str)
